# Домашняя работа 2 — качество данных

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal

DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

## Независимая загрузка

In [ ]:
decisions = pd.read_csv(
    DATA / "decisions.csv",
    dtype={"lei": "string", "county_code": "string"},
)
export = pd.read_csv(DATA / "training_export.csv", dtype="string")
original_export = export.copy(deep=True)

print("Размер decisions:", decisions.shape)
print("Размер training_export:", export.shape)
print(export.head())

## H1. Собственная функция очистки

In [ ]:
def clean_export(source):
    """Проверяет учебную выгрузку и возвращает clean, quarantine и audit."""
    original_columns = list(source.columns)
    work = source.copy(deep=True)

    audit = {
        "input": len(work),
        "replays": 0,
        "clean": 0,
        "quarantine": 0,
        "parse_errors": 0,
    }

    # Приводим технический ключ к числу только для поиска повторных доставок.
    row_key = pd.to_numeric(work["source_row"], errors="coerce")
    repeated_indexes = []

    for _, group in work.loc[row_key.notna()].groupby(row_key[row_key.notna()], sort=False):
        distinct_versions = group.drop_duplicates(subset=original_columns)
        assert len(distinct_versions) == 1, "Обнаружены конфликтующие версии одного source_row"
        if len(group) > 1:
            repeated_indexes.extend(group.index[1:].tolist())

    work = work.drop(index=repeated_indexes).copy()
    audit["replays"] = len(repeated_indexes)

    raw_values = {column: work[column].copy() for column in original_columns}

    numeric = {}
    for column in ["source_row", "action_taken", "loan_amount", "income"]:
        numeric[column] = pd.to_numeric(raw_values[column], errors="coerce")
        audit["parse_errors"] += int(
            (raw_values[column].notna() & numeric[column].isna()).sum()
        )

    county = raw_values["county_code"].str.strip()

    # Для каждой строки собираем все причины, по которым она не проходит правила.
    reasons = pd.Series("", index=work.index, dtype="string")

    def add_reason(mask, message):
        reasons.loc[mask.fillna(True)] = (
            reasons.loc[mask.fillna(True)] + message + "; "
        )

    add_reason(numeric["source_row"].notna(), "") if False else None
    add_reason(numeric["source_row"].isna(), "неверный source_row")
    add_reason(
        ~numeric["action_taken"].isin([1, 2, 3]),
        "недопустимый action_taken",
    )
    add_reason(
        numeric["loan_amount"].isna() | numeric["loan_amount"].le(0),
        "некорректная сумма кредита",
    )
    add_reason(~county.eq("11001"), "некорректная территория")
    add_reason(
        raw_values["income"].notna() & numeric["income"].isna(),
        "некорректный доход",
    )

    invalid = reasons.ne("")
    clean = work.loc[~invalid, original_columns].copy()
    quarantine = work.loc[invalid, original_columns].copy()
    quarantine["reason"] = reasons.loc[invalid].str.rstrip("; ")

    # В clean числовые поля имеют числовые типы; пропуски income остаются NaN.
    for column in ["source_row", "action_taken"]:
        clean[column] = numeric[column].loc[~invalid].astype("Int64")
    clean["loan_amount"] = numeric["loan_amount"].loc[~invalid].astype(float)
    clean["income"] = numeric["income"].loc[~invalid].astype(float)
    clean["county_code"] = county.loc[~invalid]

    audit["clean"] = len(clean)
    audit["quarantine"] = len(quarantine)

    assert audit["input"] == (
        audit["replays"] + audit["clean"] + audit["quarantine"]
    )
    return clean, quarantine, audit

clean, quarantine, audit = clean_export(export)
print("Журнал:", audit)
print("Размер clean:", clean.shape)
print("Размер quarantine:", quarantine.shape)
quarantine

## H2. Год и известный доход

In [ ]:
year_ok = clean["activity_year"].eq("2023").fillna(False).all()
total_rows = len(clean)
known_income = int(clean["income"].notna().sum())
unknown_income = int(clean["income"].isna().sum())
mean_income = clean["income"].mean()

assert year_ok
assert total_rows == known_income + unknown_income
assert audit["input"] == audit["replays"] + total_rows + len(quarantine)
assert_frame_equal(export, original_export)

print("Год везде 2023:", year_ok)
print("Всего подготовленных записей:", total_rows)
print("Известный доход:", known_income)
print("Неизвестный доход:", unknown_income)
print(f"Средний известный доход: {mean_income:.2f} тысяч USD")
print("Баланс строк сходится; исходная таблица не изменена.")

## H3. Проверка отчётного года

In [ ]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_check = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("Проверка года после изменения первой строки:", year_check)

assert year_check is False
assert_frame_equal(export, original_export)
print("Исходная таблица export осталась без изменений.")

## Дополнительные задания — по желанию

H+1 и H+2 не выполнялись. Базовая работа H1–H3 выполнена.